In [ ]:
for sub in HELDOUT_SUBS:
    assert sub in enc and sub in dec, f'missing scores for sub-{sub:02d}'
    S = fuse(enc[sub], dec[sub], W_ENC)
    r = ranked_ids(S)
    path = save_submission(r, sub)
    # independent re-check from disk
    on_disk = np.load(path)
    check_submission(on_disk)
    # row count of distinct top-1 picks is a quick sanity signal (model is not predicting one id)
    uniq = len(np.unique(on_disk[:, 0]))
    print(f'sub-{sub:02d}  wrote {path}  distinct top-1 ids = {uniq}/102')

## Write sub-05 / sub-06 submissions

In [ ]:
def score_rows(name, S_by_sub):
    rows = []
    for s in DEV_SUBS:
        if s not in S_by_sub: continue
        rows.append({'stream': name, 'sub': s,
                     **topk(ranked_ids(S_by_sub[s]), load_test_labels(s))})
    return rows

fused = {s: fuse(enc[s], dec[s], W_ENC) for s in SUBS if s in enc and s in dec}

import pandas as pd
rows = []
rows += score_rows('encoding', enc)
rows += score_rows('decoder',  dec)
rows += score_rows('fused',    fused)
df = pd.DataFrame(rows)
print(df.pivot_table(index='stream', values=['top1', 'top5', 'median_rank'],
                     aggfunc={'top1': 'mean', 'top5': 'mean', 'median_rank': 'median'}))

# sub-01 round trip through the file format
r1 = ranked_ids(fused[1])
in_memory = topk(r1, load_test_labels(1))
path = save_submission(r1, 1, out_dir='./submissions_check')
on_disk = topk(np.load(path), load_test_labels(1))
print('sub-01 in-memory:', in_memory)
print('sub-01 on-disk  :', on_disk)
assert in_memory == on_disk, 'submission round-trip changed the score'

## Per-subject final scores and sub-01 round-trip

Print top-1/top-5 for each stream alone and for the fused output on sub-01..04. Then write a `sub-01` ranking file and reload it to confirm the on-disk → score pipeline matches the in-memory score (this is the exact code path we'll use for 05/06).

In [ ]:
def fuse(S_enc, S_dec, w):
    return w * rank_norm(S_enc) + (1 - w) * rank_norm(S_dec)

def mean_top1(w):
    scores = []
    for s in DEV_SUBS:
        if s not in enc or s not in dec: continue
        S = fuse(enc[s], dec[s], w)
        scores.append(topk(ranked_ids(S), load_test_labels(s))['top1'])
    return float(np.mean(scores)), scores

ws = np.linspace(0, 1, 21)
grid = [(w, *mean_top1(w)) for w in ws]
for w, mt1, _ in grid:
    print(f'w_enc={w:.2f}  mean top1={mt1:.3f}')
best = max(grid, key=lambda x: x[1])
W_ENC = float(best[0])
print(f'\nchosen w_enc={W_ENC:.2f}  mean top1={best[1]:.3f}  per-sub={best[2]}')

## Pick fusion weight on sub-01..04

Grid over `w ∈ [0, 1]` where `S_fused = w * rank(S_enc) + (1-w) * rank(S_dec)`; pick the `w` that maximises mean top-1 across sub-01..04.

In [ ]:
def load_stream(name):
    out = {}
    for s in SUBS:
        p = f'{SCORES_DIR}/{name}_sub-{s:02d}.npz'
        if not os.path.exists(p):
            print('missing', p); continue
        z = np.load(p)
        assert (z['test_ids'] == TEST_IDS).all(), f'{p}: test_ids mismatch'
        assert z['S'].shape == (102, 102), z['S'].shape
        out[s] = z['S'].astype(np.float32)
    return out

enc = load_stream('encoding')
dec = load_stream('decoder')
print('enc subs:', sorted(enc), 'dec subs:', sorted(dec))

def rank_norm(S):
    """Per-row rank in [0, 1]; higher = better match."""
    r = np.argsort(np.argsort(-S, axis=1), axis=1).astype(np.float32)
    return 1.0 - r / (S.shape[1] - 1)

## Load and normalise both score streams

In [ ]:
import os, glob
import numpy as np

TEST_DIR      = './tst'
TEST_STIM_DIR = './tst_stim'
SCORES_DIR    = './scores'
SUBMIT_DIR    = './submissions'
os.makedirs(SUBMIT_DIR, exist_ok=True)

SUBS         = [1, 2, 3, 4, 5, 6]
DEV_SUBS     = [1, 2, 3, 4]
HELDOUT_SUBS = [5, 6]
TEST_IDS     = np.arange(1001, 1103)

def load_test_labels(sub):
    assert sub in DEV_SUBS
    return np.load(f'{TEST_STIM_DIR}/sub-{sub:02d}_tst_stim.npy').astype(int)

def ranked_ids(S, video_ids=TEST_IDS):
    return np.asarray(video_ids)[np.argsort(-S, axis=1)]

def topk(ranked, truth, ks=(1, 5)):
    hits = ranked == np.asarray(truth)[:, None]
    assert hits.any(1).all()
    rank = hits.argmax(1) + 1
    out = {f'top{k}': float((rank <= k).mean()) for k in ks}
    out['median_rank'] = float(np.median(rank))
    return out

def check_submission(ranked):
    ranked = np.asarray(ranked)
    assert ranked.shape == (102, 102), ranked.shape
    assert np.issubdtype(ranked.dtype, np.integer), ranked.dtype
    assert (np.sort(ranked, 1) == TEST_IDS).all(), 'each row must be a permutation of 1001..1102'

def save_submission(ranked, sub, out_dir=SUBMIT_DIR):
    check_submission(ranked)
    path = f'{out_dir}/sub-{sub:02d}_ranked_candidate_list.npy'
    np.save(path, np.asarray(ranked, dtype=np.int64))
    return path

# 06 — Ensemble and submission

Fuse the encoding-model similarity (`scores/encoding_sub-XX.npz` from 04) with the contrastive-decoder similarity (`scores/decoder_sub-XX.npz` from 05) into one ranking per subject. Pick fusion weights on sub-01..04 (CV score), apply to sub-05/06, and write the required `.npy` files.

Steps:
1. Load both score streams per subject.
2. **Per-row rank-normalise** each stream (robust to scale differences; identical to the identification task's own metric).
3. Fuse by weighted mean. Weights are chosen on sub-01..04 top-1 and reused on 05/06.
4. Round-trip sub-01's ranking through the exact submission-file path and confirm the score matches the in-memory score.
5. Write `submissions/sub-05_ranked_candidate_list.npy` and `submissions/sub-06_ranked_candidate_list.npy`, each `(102, 102)` int64 with rows in `load_test(sub)` order.

The submission writer asserts shape, dtype, and that each row is a permutation of 1001..1102.